In [ ]:
#hypothesis tests
import pandas as pd
path='http://datayyy.com/data_pickle/'
#获取一个存放美国季度 GDP 数据
infile=path+'usGDPquarterly.pkl' 
GDP=pd.read_pickle(infile)
#美国 Fama-French 3 因子月度金融数据集
infile2=path+'ff3monthly.pkl' 
ff3=pd.read_pickle(infile2)
print(ff3.head())

#stats.norm 是 Python 里标准正态分布的工具，标准正态分布 N (0,1)：均值 = 0，标准差 = 1，钟形对称曲线
#pdf = 概率密度、cdf = 累积概率、ppf = 分位数（cdf 逆运算）
#pdf = 看图有多高(给定x看对应的概率),cdf = 算左边面积（给定x看左侧）,ppf = 已知面积反查x值
#np.random.normal 用于生成服从正态分布的随机数，可指定均值、标准差与数量
import numpy as np
import scipy.stats as stats
x=0.5
y=stats.norm.cdf(x) 

#alpha=5%=95%CI 绘制标准正态分布概率密度曲线
#vlines画垂直临界线，axhline画区间标记线，figtext在图中间写 95% CI
import matplotlib.pyplot as plt
alpha=0.05
x=np.arange(-3,3,0.15)
y=stats.norm.pdf(x)
plt.plot(x,y)
#并标出α=0.05 对应的双侧临界值（≈±1.96），中间区域就是95% 置信区间
z1=stats.norm.ppf(alpha) #已知面积求x
z2=stats.norm.ppf(1-alpha)
#plt.vlines(x坐标, 起点y, 终点y, 颜色, 线型)
plt.vlines(z1, 0,0.34,color='green',ls=':',lable='partical') #label 给这条线图起个名字，用来在右上角显示图例（legend
plt.vlines(z2, 0,0.34,color='green',ls=':')
#plt.axhline(y坐标, xmin=左, xmax=右)  # x是0~1比例
plt.axhline(y=0.25,xmin=0.25,xmax=0.75,ls='-',color='red')
#plt.figtext(横坐标比例, 纵坐标比例, "文字")
plt.figtext(0.4, 0.5, '95%CI')
plt.title('normal distribution pdf')
plt.xlable('values')
plt.ylabel('density')
plt.show()

In [ ]:
#example of two-sided tests
#模拟一组正态数据，计算总体均值的 95% 双侧置信区间
mean=0.15
std=0.03
n=300
## 生成300个正态随机数
x=np.random.normal(mean,std,n)
alpha=0.05
## 标准误 = 标准差/√n
stderr=std/np.sqrt(n)
z=stats.norm.ppf(alpha)
marginoferr=abs(z)*stderr #边际误差
lowerbound=mean-marginoferr # 置信区间下限
upperbound=mean+marginoferr
print(lowerbound,upperbound)

In [ ]:
#最小二乘法（OLS）做一元线性回归，输出截距和斜率两个回归系数
import statsmodels.api as sm
y=[1,2,3,4,2,3,4]
x=range(1,len(y)+1)
x=sm.add_constant(x)
results=sm.OLS(y,x).fit()
print(results.params) ## 输出：截距a 和 斜率b

'''df=pd.read_pickle('ibm.pkl')
df['ret']=df['Close'].pct_change()
df.ret.mean()'''

In [ ]:
#conversion between percentage(return) and log return
#用 ln 把普通收益率转对数收益率，用 e 指数还原回普通收益率
from math import exp,log
r=0.1
logret=log(r+1)
r2=exp(logret)-1
print(r2)


In [ ]:
#test the equal variances of two samples
#两样本方差齐性 F 检验，用来判断两组数据的方差是否相等
infile='http://datayyy.com/data_csv/sleep.csv'
df=pd.read_csv(infile)
group1=df[df.group==1]
group2=df[df.group==2]
var1=group1.extra.var() #取出第 1 组的 extra 这一列数据计算方差
var2=group2.extra.var()
#原假设 H0​：两组方差相等
#f越接近1，方差越接近
fvalue=max(var1,var2)/min(var1,var2)
df1=group1.shape[0]-1 #自由度=样本量-1
df2=group2.shape[0]-1
alpha=0.05
#stats.f.ppf(1-alpha, df1, df2) 是根据 F 分布、两个自由度和显著性水平，计算并返回 F 检验的右侧临界值，用于判断是否拒绝方差相等的原假设。
fcritical=stats.f.ppf(1-alpha,df1,df2)
print(fvalue)
if(fvalue>fcritical):
    print('reject h0')
else:
    print('accept h0')
    
    
#the second and easier method:its called levene function
#stats.levene(组1, 组2) 是直接检验两组数据方差是否相等的函数，自动算出 p 值
stats.levene(group1.extra,group2.extra)
#pvalue=np.float64(0.6243742854148879))大于0.05，接受原假设

#test of equal means
group1.extra.mean()
group2.extra.mean()
#stats.ttest_ind(组1数据, 组2数据)= 检验两组均值是否相等,算出t统计量、p值、自由度df,p 值 > 0.05 → 接受 H₀ → 两组均值没差异
stats.ttest_ind(group1.extra, group2.extra)
#TtestResult(statistic=np.float64(-1.8608134674868524), pvalue=np.float64(0.07918671421593829), df=np.float64(18.0))



In [ ]:
#to test if ibm has the same return as sp500
df1=pd.read_pickle('ibm')
df2=pd.read_pickle('sp500')
df1['ibmret']=df1['close'].pct_change()
df2['sp500ret']=df2['close'].pct_change()
df=df1.merge(df2,left_on=df1.index,right_on=df2.index)
df=df.dropna()
stats.ttset_ind(a=df.ibmret,b=df.sp500ret) #均值是否相等
stats.levene(df.ibmret,df.sp500ret) #方差是否相等

#constant annual variance 
#use ibm as the example 检验 IBM 年度方差是否相等
df1['year']=df.index.year # 从日期里提取年份
#test the year of 2012-2014
ret2012=df1[df1.year==2012].dropna() #取出每年的收益
ret2013=df1[df1.year==2013].dropna()
ret2014=df1[df1.year==2014].dropna()
stats.levene(ret2012.ibmret,ret2013.ibmret,ret2014.ibmret) #方差是否相等

In [ ]:
#use ibm daily returns auto-correlated 检验 IBM 日收益率是否存在自相关
results=sm.tsa.acf(df1.ibmret.dropna())
#sm.tsa.acf() 是时间序列自相关函数，用于计算同一组数据在不同滞后期的相关系数，衡量当期收益率与前 1 期、前 2 期…… 历史收益率之间的关联强弱，
#判断序列是否存在时序自相关性，金融中常用来检验股票收益率是否可由历史收益预测。
#[1.0,  0.04,  -0.01,  0.02,  0.00, ...]今天vs今天；今天vs昨天....
print(results)

In [ ]:
#use ibm daily returns auto-correlated 检验 IBM 日收益率是否存在自相关
results=sm.tsa.acf(df1.ibmret.dropna())
#sm.tsa.acf() 是时间序列自相关函数，用于计算同一组数据在不同滞后期的相关系数，衡量当期收益率与前 1 期、前 2 期…… 历史收益率之间的关联强弱，
#判断序列是否存在时序自相关性，金融中常用来检验股票收益率是否可由历史收益预测。
#[1.0,  0.04,  -0.01,  0.02,  0.00, ...]今天vs今天；今天vs昨天....
print(results)

#chicken or egg first
infile='http://datayyy.com/data_pickle/chickEgg.pickle'
df=pd.read_pickle(infile)
#给 chicken 做滞后1、2、3期（昨天、前天、大前天的鸡)
df['chicken_lag1']=df.chicken.shift(1)
df['chicken_lag2']=df.chicken.shift(2)
df['chicken_lag3']=df.chicken.shift(3)
# 给 egg 做滞后1、2、3期（昨天、前天、大前天的蛋）
df['egg_lag1']=df.egg.shift(1)
df['egg_lag2']=df.egg.shift(2)
df['egg_lag3']=df.egg.shift(3)
df=df.dropna()
#用过去 3 天的蛋 → 预测今天的蛋,蛋自己的过去能不能解释现在
y=df['egg']
x=df[['egg_lag1','egg_lag2','egg_lag3']]
x=sm.add_constant(x)
results=sm.OLS(y, x).fit()
print(results.summary())
#用过去 3 天的鸡 → 预测今天的鸡,鸡自己的过去能不能解释现在
y=df['chicken']
x=df[['chicken_lag1','chicken_lag2','chicken_lag3']]
x=sm.add_constant(x)
results=sm.OLS(y, x).fit()
print(results.summary())
#R-squared：越高说明过去越能预测现在；P>|t|：<0.05 表示滞后项显著有效

In [ ]:
#use ibm daily returns auto-correlated 检验 IBM 日收益率是否存在自相关
results=sm.tsa.acf(df1.ibmret.dropna())
#sm.tsa.acf() 是时间序列自相关函数，用于计算同一组数据在不同滞后期的相关系数，衡量当期收益率与前 1 期、前 2 期…… 历史收益率之间的关联强弱，
#判断序列是否存在时序自相关性，金融中常用来检验股票收益率是否可由历史收益预测。
#[1.0,  0.04,  -0.01,  0.02,  0.00, ...]今天vs今天；今天vs昨天....
print(results)

#chicken or egg first
infile='http://datayyy.com/data_pickle/chickEgg.pickle'
df=pd.read_pickle(infile)
#给 chicken 做滞后1、2、3期（昨天、前天、大前天的鸡)
df['chicken_lag1']=df.chicken.shift(1)
df['chicken_lag2']=df.chicken.shift(2)
df['chicken_lag3']=df.chicken.shift(3)
# 给 egg 做滞后1、2、3期（昨天、前天、大前天的蛋）
df['egg_lag1']=df.egg.shift(1)
df['egg_lag2']=df.egg.shift(2)
df['egg_lag3']=df.egg.shift(3)
df=df.dropna()
#用过去 3 天的蛋 → 预测今天的蛋,蛋自己的过去能不能解释现在
y=df['egg']
x=df[['egg_lag1','egg_lag2','egg_lag3']]
x=sm.add_constant(x)
results=sm.OLS(y, x).fit()
print(results.summary())
#用过去 3 天的鸡 → 预测今天的鸡,鸡自己的过去能不能解释现在
y=df['chicken']
x=df[['chicken_lag1','chicken_lag2','chicken_lag3']]
x=sm.add_constant(x)
results=sm.OLS(y, x).fit()
print(results.summary())
#R-squared：越高说明过去越能预测现在；P>|t|：<0.05 表示滞后项显著有效

#test of january effect
df1['logret']=np.log(df1['Close'].pct_change()+1)
df['YYYYMM']=df.index.year*100+df.index.month
retm=np.exp(df['logret'].groupby(df['YYYYMM']).sum())-1
retm=pd.DataFrame(retm)
retm.columns=['retm']
#这段代码通过计算股票月度对数收益率，
#分离出1 月收益率和非 1 月收益率，
#再用独立样本 t 检验检验两者均值是否相等，
#以此验证金融市场著名的一月效应（1 月收益是否显著更高）；
#看结果只需关注 p 值，p<0.05 则存在一月效应，p>0.05 则不存在。
janret=[]
nonjanret=[]
for i in retm.index:
    mm=i-int(i/1e2)*100
    ret=retm[retm.index==i].retm
    if mm==1:
        print(i)
        janret.append(ret.values)
    else:
        nonjanret.append(ret.values)
janret=pd.DataFrame(janret)
nonjanret=pd.DataFrame(nonjanret)
results=stats.ttest_ind(janret,nonjanret)
print(results)
#这段代码将股票月度收益率分为偶数月份（2/4/6…）和奇数月份（1/3/5…)两组，
#通过独立样本 t 检验检验两组收益均值是否相等，输出 p 值 <0.05 说明奇偶月份收益有显著差异，p>0.05 则无差异。
evenret=[]
oddret=[]
for i in retm.index:
    mm=i-int(i/1e2)*100
    ret=retm[retm.index==i].retm
    if mm%2==0:
        print(i)
        evenret.append(ret.values)
    else:
        oddret.append(ret.values)
evenret=pd.DataFrame(evenret)
oddret=pd.DataFrame(oddret)
results=stats.ttest_ind(evenret,oddret)
print(results)

